<a href="https://colab.research.google.com/github/MayerT1/AG_Rapid_Prototypes/blob/main/csda_Planet_fetcher_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSDA Planet Imagery Downloader
Downloads Planet imagery from NASA CSDA over a defined AOI for a list of dates.

**Run cells top to bottom. Edit Cell 2 to set your dates and AOI.**

In [8]:
# Cell 1: Install dependencies
!pip install -q requests tqdm git+https://github.com/nasa-impact/csda-client.git httpx

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [9]:
# Cell 2: CONFIGURATION — Edit this cell

# ── Dates to search ──────────────────────────────────────────────
DATES = [
    "2023-06-01",
    "2023-06-15",
    "2023-07-01",
]

# ── AOI ──────────────────────────────────────────────────────────
# Option A: paste a GeoJSON geometry inline (Polygon or MultiPolygon)
# Set to None to load from a file instead (Option B)
AOI_GEOMETRY = {
    "type": "Polygon",
    "coordinates": [[
        [-104.1, 40.0],
        [-95.3,  40.0],
        [-95.3,  43.0],
        [-104.1, 43.0],
        [-104.1, 40.0]
    ]]
}

# Option B: path to an uploaded .geojson file
# Upload your file via the Colab file browser, then set AOI_GEOMETRY = None
GEOJSON_PATH = "aoi.geojson"

# ── Collection ───────────────────────────────────────────────────
# Options: "planet", "maxar-sdx", "blacksky", "capellaspace", etc.
COLLECTION = "planet"

# ── Asset preference (first match wins) ──────────────────────────
PREFERRED_ASSETS = ["ortho_analytic_8b_sr", "ortho_analytic_8b", "ortho_visual", "analytic", "visual"]

# ── Max STAC results per date query ──────────────────────────────
LIMIT = 100

# ── Download folder ───────────────────────────────────────────────
DOWNLOAD_DIR = "/content/csda_downloads"

In [10]:
# Cell 3: Authenticate
import getpass, base64, json, requests, os
from pathlib import Path
from tqdm.notebook import tqdm
from csda_client import CsdaClient
from httpx import BasicAuth

username = input("Earthdata Username: ")
password = getpass.getpass("Earthdata Password: ")

# Get bearer token for STAC search
credentials = base64.b64encode(f"{username}:{password}".encode()).decode()
r = requests.post(
    "https://urs.earthdata.nasa.gov/api/users/find_or_create_token",
    headers={"Authorization": f"Basic {credentials}"}
)
assert r.status_code == 200, f"Auth failed: {r.text}"
TOKEN = r.json()["access_token"]
print("Token acquired")

# Login csda-client for downloads
csda = CsdaClient()
csda.login(BasicAuth(username, password))
print("CSDA client ready:", csda.verify())

Earthdata Username: tjm0042
Earthdata Password: ··········
Token acquired
CSDA client ready: Hello tjm0042, you have a valid token!


In [11]:
# Cell 4: Load AOI

def load_geometry(inline_geom, path):
    if inline_geom is not None:
        print(f"Using inline AOI ({inline_geom['type']})")
        return inline_geom
    with open(path) as f:
        gj = json.load(f)
    if gj["type"] == "FeatureCollection":
        geom = gj["features"][0]["geometry"]
    elif gj["type"] == "Feature":
        geom = gj["geometry"]
    else:
        geom = gj
    print(f"Loaded AOI from {path} ({geom['type']})")
    return geom

geometry = load_geometry(AOI_GEOMETRY, GEOJSON_PATH)
print("AOI ready")

Using inline AOI (Polygon)
AOI ready


In [12]:
# Cell 5: Search STAC
STAC_URL = "https://csdap.earthdata.nasa.gov/stac/search"
headers = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

all_features = []

for date in DATES:
    print(f"\nQuerying {date}...")
    payload = {
        "intersects": geometry,
        "datetime": f"{date}T00:00:00Z/{date}T23:59:59Z",
        "collections": [COLLECTION],
        "limit": LIMIT,
    }
    seen_ids = set()
    page = 1

    while True:
        r = requests.post(STAC_URL, headers=headers, json=payload)
        if r.status_code != 200:
            print(f"  Error {r.status_code}: {r.text}")
            break
        resp = r.json()
        features = resp.get("features", [])
        if not features:
            break
        new = [f for f in features if f.get("id") not in seen_ids]
        if not new:
            break
        for f in new:
            f["_queried_date"] = date
            seen_ids.add(f.get("id"))
        all_features.extend(new)
        print(f"  Page {page}: {len(new)} items")

        next_links = [l for l in resp.get("links", []) if l.get("rel") == "next"]
        if not next_links:
            break
        from urllib.parse import urlparse, parse_qs
        qs = parse_qs(urlparse(next_links[0].get("href", "")).query)
        if "token" in qs:
            payload["token"] = qs["token"][0]
        elif "page" in qs:
            payload["page"] = int(qs["page"][0])
        else:
            break
        page += 1

print(f"\nTotal items found: {len(all_features)}")
with open("/content/csda_results.geojson", "w") as f:
    json.dump({"type": "FeatureCollection", "features": all_features}, f, indent=2)
print("Search results saved to /content/csda_results.geojson")


Querying 2023-06-01...
  Page 1: 61 items

Querying 2023-06-15...
  Page 1: 100 items

Querying 2023-07-01...
  Page 1: 32 items

Total items found: 193
Search results saved to /content/csda_results.geojson


In [13]:
# Cell 6: Preview results
print(f"{'ID':<50} {'Date':<12} Assets")
print("-" * 90)
for feat in all_features[:20]:
    fid = feat.get("id", "unknown")[:48]
    date = feat.get("_queried_date", "")
    keys = list(feat.get("assets", {}).keys())
    print(f"{fid:<50} {date:<12} {keys}")
if len(all_features) > 20:
    print(f"... and {len(all_features) - 20} more")

ID                                                 Date         Assets
------------------------------------------------------------------------------------------
PSScene-20230601_172141_51_24a4                    2023-06-01   ['thumbnail', 'basic_udm2', 'ortho_udm2', 'ortho_visual', 'json_metadata', 'basic_analytic_8b', 'ortho_analytic_8b', 'ortho_analytic_8b_sr', 'basic_analytic_4b_rpc', 'basic_analytic_8b_xml', 'ortho_analytic_8b_xml']
PSScene-20230601_172139_32_24a4                    2023-06-01   ['thumbnail', 'basic_udm2', 'ortho_udm2', 'ortho_visual', 'json_metadata', 'basic_analytic_8b', 'ortho_analytic_8b', 'ortho_analytic_8b_sr', 'basic_analytic_4b_rpc', 'basic_analytic_8b_xml', 'ortho_analytic_8b_xml']
PSScene-20230601_171501_81_227a                    2023-06-01   ['thumbnail', 'basic_udm2', 'ortho_udm2', 'json_metadata', 'basic_analytic_8b', 'ortho_analytic_8b', 'ortho_analytic_8b_sr', 'basic_analytic_4b_rpc', 'basic_analytic_8b_xml', 'ortho_analytic_8b_xml']
PSScene-202306

In [ ]:
# Cell 7: Download imagery
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

def pick_asset(assets, preferred):
    for key in preferred:
        if key in assets:
            return key, assets[key]
    if assets:
        key = next(iter(assets))
        return key, assets[key]
    return None, None

ok, fail, skip = 0, 0, 0

for feat in all_features:
    item_id = feat.get("id", "unknown")
    date = feat.get("_queried_date", "nodate")
    collection = feat.get("collection", COLLECTION)
    asset_key, asset = pick_asset(feat.get("assets", {}), PREFERRED_ASSETS)

    if not asset_key:
        fail += 1
        continue

    href = asset.get("href", "")
    ext = Path(href.split("?")[0]).suffix or ".tif"
    dest = Path(DOWNLOAD_DIR) / date / f"{item_id}_{asset_key}{ext}"
    dest.parent.mkdir(parents=True, exist_ok=True)

    if dest.exists():
        skip += 1
        continue

    print(f"  Downloading {date}/{item_id}_{asset_key}{ext}")
    try:
        csda.download(collection, item_id, asset_key, dest)
        ok += 1
        print(f"    Done")
    except Exception as e:
        print(f"    Error: {e}")
        fail += 1

print(f"\nDownloaded: {ok} | Skipped: {skip} | Failed: {fail}")
print(f"Files in: {DOWNLOAD_DIR}")

In [ ]:
# Cell 8 (optional): Zip and download to your machine
import shutil
from google.colab import files

zip_path = "/content/csda_imagery"
shutil.make_archive(zip_path, "zip", DOWNLOAD_DIR)
print(f"Archive ready: {zip_path}.zip")
files.download(f"{zip_path}.zip")